# Clip Tuvalu DEM and Population Data to Funafuti — Boundary-based workflow

This preprocessing notebook clips:

1. the Tuvalu ASTER GDEM source raster, and
2. the WorldPop 2020 population raster,

to the **same Funafuti administrative study-area boundary** fetched from
OpenStreetMap.

Tuvalu is commonly described as nine islands/atolls/reef islands. This project
does **not** model all nine individually; detailed elevation/population exposure
analysis is scoped to **Funafuti**.

Using the same boundary does **not** mean ASTER and WorldPop share the same
native CRS or pixel grid. They retain their native spatial properties after
clipping; `01_flood_exposure_analysis.ipynb` later reprojects/resamples WorldPop
onto the DEM's exact grid for overlay analysis.

**Methodology statement:**
> The ASTER DEM and WorldPop population raster were clipped to the same Funafuti
> administrative study-area boundary prior to validation and exposure analysis.

**Prerequisite:** this notebook needs internet access to retrieve the Funafuti
boundary through `osmnx`/OpenStreetMap. If the processed Funafuti rasters already
exist and have been validated, this clipping notebook does not need to be rerun
every time the website is rebuilt.

Use **Restart Kernel and Run All** when executing this notebook from scratch.


## 1. Imports and paths

In [ ]:
import osmnx as ox
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_geom
from shapely.geometry import mapping
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Portable project root: works when Jupyter is launched from the repo root
# or from the notebooks/ directory. No machine-specific absolute paths.
PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
DEM_SRC_PATH = PROJECT_ROOT / "data" / "raw" / "TUV_2019_DEM1arcsec_ASTGTMv3.tif"
DEM_OUTPUT_PATH = PROJECT_ROOT / "data" / "raw" / "tuvalu_dem_funafuti_admin.tif"

print("DEM path:", DEM_SRC_PATH, "exists:", DEM_SRC_PATH.exists())

## 2. Fetch Funafuti's administrative boundary from OpenStreetMap

`osmnx.geocode_to_gdf` sends a request to Nominatim and returns the
geometry matching "Funafuti, Tuvalu" as a GeoDataFrame (a polygon already,
no need to manually convert lines to polygons).

In [ ]:
boundary_gdf = ox.geocode_to_gdf("Funafuti, Tuvalu")
print(boundary_gdf[["display_name", "geometry"]])
print("\nOriginal CRS:", boundary_gdf.crs)

# Quick sanity plot of the shape (should be a narrow, diagonally-oriented atoll outline)
fig, ax = plt.subplots(figsize=(5, 5))
boundary_gdf.plot(ax=ax, edgecolor="crimson", facecolor="none", linewidth=2)
ax.set_title("Funafuti administrative boundary (from OSM)")
plt.show()

**Check**: if the plot above shows a narrow, diagonally-oriented shape
(consistent with the atoll outline seen earlier in QGIS's OSM Place Search),
the fetch worked. If it shows a huge extent spanning the whole Pacific (e.g.
longitude from -180 to 180), Nominatim likely matched the country level
("Tuvalu") rather than Funafuti itself -- try adding a more specific
qualifier to the query, e.g. `ox.geocode_to_gdf("Funafuti, Tuvalu",
which_result=2)` to try a different match, or query a more specific name
such as `"Funafuti Conservation Area, Tuvalu"`.

## 3. Reproject the boundary to the DEM's CRS, and clip the DEM

In [ ]:
with rasterio.open(DEM_SRC_PATH) as src:
    dem_crs = src.crs
    print("DEM CRS:", dem_crs)

    # Reproject the boundary to the DEM's CRS (EPSG:3832)
    boundary_proj = boundary_gdf.to_crs(dem_crs)
    geom_dem = [mapping(boundary_proj.geometry.iloc[0])]

    # Clip the DEM using the administrative boundary as a mask; crop=True
    # automatically tightens the output extent to the boundary's bounding box
    dem_out_image, dem_out_transform = mask(src, geom_dem, crop=True, nodata=src.nodata)
    dem_out_meta = src.meta.copy()
    dem_out_meta.update({
        "height": dem_out_image.shape[1],
        "width": dem_out_image.shape[2],
        "transform": dem_out_transform,
    })

print("Cropped size:", dem_out_image.shape[2], "x", dem_out_image.shape[1], "pixels")

## 4. Sanity checks (area, elevation range)

In [ ]:
dem_band = dem_out_image[0]
dem_nodata = dem_out_meta["nodata"]

dem_valid = dem_band[dem_band != dem_nodata] if dem_nodata is not None else dem_band
dem_land = dem_valid[dem_valid > 0]

dem_pixel_area_m2 = abs(dem_out_transform.a * dem_out_transform.e)
dem_land_area_km2 = dem_land.size * dem_pixel_area_m2 / 1e6

print(f"Land pixel count: {dem_land.size:,}")
print(f"Land area (rough): {dem_land_area_km2:.2f} km2  (reference: Funafuti is ~2.4 km2)")
if dem_land.size > 0:
    print(f"Land elevation range: {float(dem_land.min()):.2f}m ~ {float(dem_land.max()):.2f}m "
          f"(reference: Tuvalu's highest point is ~4.6m -- ASTER often shows elevated "
          f"noise on atolls like this, so don't be alarmed if it's a bit higher)")

# Quick visualisation of the cropped result
fig, ax = plt.subplots(figsize=(6, 6))
dem_display = np.where(dem_band == dem_nodata, np.nan, dem_band)
im = ax.imshow(dem_display, cmap="terrain")
ax.set_title("Clipped DEM: Funafuti (administrative boundary)")
plt.colorbar(im, ax=ax, label="Elevation (m)")
plt.show()

**Check**: the computed land area should be on the order of 2-3 km2 (not
tens or hundreds). If it's far off, the boundary fetched in Step 2 likely
still includes a large area of administered sea/EEZ -- see the check note
in Step 2 and retry with a more precise query match.

## 5. Save the cropped DEM

In [ ]:
with rasterio.open(DEM_OUTPUT_PATH, "w", **dem_out_meta) as dst:
    dst.write(dem_out_image)

print(f"Saved: {DEM_OUTPUT_PATH}")
print(f"File size: {DEM_OUTPUT_PATH.stat().st_size / 1024:.1f} KB")

## 6. DEM output and downstream use

The cropped DEM is saved as:

`data/raw/tuvalu_dem_funafuti_admin.tif`

Do not manually force the population raster to the DEM's CRS/grid here. The
formal analysis notebook performs that alignment explicitly and reproducibly
after both source rasters have been clipped to the same study-area boundary.


---

## 7. Clip the WorldPop population raster to the same Funafuti boundary

Re-uses the exact same administrative boundary from Step 2 above, so the
population raster and the DEM share the same spatial reference — this is
the whole point of using an administrative boundary rather than the
atoll's natural extent (see the note at the top of this notebook).

**Source file**: `tuv_pop_2020_CN_100m_R2025A_v1.tif` (WorldPop, 2020
Population Counts, 100m, R2025A v1 — see README.md for the full citation).

In [ ]:
POP_SRC_PATH = PROJECT_ROOT / "data" / "raw" / "tuv_pop_2020_CN_100m_R2025A_v1.tif"
POP_OUTPUT_PATH = PROJECT_ROOT / "data" / "raw" / "tuvalu_pop_funafuti_admin.tif"

print("Population raster path:", POP_SRC_PATH, "exists:", POP_SRC_PATH.exists())

with rasterio.open(POP_SRC_PATH) as src:
    print("Population raster CRS:", src.crs)
    print("Population raster resolution:", src.res)
    print("Population raster bounds:", src.bounds)

**CRS note:** WorldPop is normally supplied in EPSG:4326, while the ASTER
Funafuti DEM used in this project is in a projected CRS. That native CRS
difference is expected.

The cell below defensively reprojects the **boundary geometry** into the
population raster's native CRS before clipping. It does not reproject the
population raster itself. Raster-to-raster alignment occurs later in
`01_flood_exposure_analysis.ipynb`.


In [ ]:
with rasterio.open(POP_SRC_PATH) as src:
    pop_crs = src.crs

    boundary_pop_proj = boundary_gdf.to_crs(pop_crs)
    geom_pop = [mapping(boundary_pop_proj.geometry.iloc[0])]

    pop_out_image, pop_out_transform = mask(src, geom_pop, crop=True, nodata=src.nodata)
    pop_out_meta = src.meta.copy()
    pop_out_meta.update({
        "height": pop_out_image.shape[1],
        "width": pop_out_image.shape[2],
        "transform": pop_out_transform,
    })

print("Cropped population raster size:", pop_out_image.shape[2], "x", pop_out_image.shape[1], "pixels")

## 8. Sanity check and visualise the cropped population raster

In [ ]:
pop_band = pop_out_image[0]
pop_nodata = pop_out_meta["nodata"]

pop_valid = pop_band[pop_band != pop_nodata] if pop_nodata is not None else pop_band
pop_valid = pop_valid[pop_valid > 0]

total_population = float(pop_valid.sum())
print(f"Total population within the clipped extent: {total_population:,.0f}")
print("Reference: Funafuti's population was 6,320 at the 2017 census (~60% of Tuvalu's national total)")
print("Note: this is a 2020 WorldPop estimate, not the 2017 census figure -- some difference is expected")

fig, ax = plt.subplots(figsize=(6, 6))
pop_display = np.where(pop_band == pop_nodata, np.nan, pop_band)
im = ax.imshow(pop_display, cmap="magma")
ax.set_title("Clipped population raster: Funafuti (WorldPop 2020)")
plt.colorbar(im, ax=ax, label="Estimated people per pixel")
plt.show()

**Population sanity check:** the native clipped WorldPop total should be in
the thousands. For reference, Funafuti's **2017 usual-resident census
population was 6,320**.

WorldPop 2020 is a modelled population estimate rather than a census headcount,
so this comparison is a validation benchmark only. Close agreement supports the
study-area clip, but does not make the two datasets equivalent.


## 9. Save the cropped population raster

In [ ]:
with rasterio.open(POP_OUTPUT_PATH, "w", **pop_out_meta) as dst:
    dst.write(pop_out_image)

print(f"Saved: {POP_OUTPUT_PATH}")
print(f"File size: {POP_OUTPUT_PATH.stat().st_size / 1024:.1f} KB")

## 10. Final next steps

Once both processed rasters exist:

1. Run `notebooks/03_validate_data.ipynb`.
2. If all blocking checks pass, run
   `notebooks/01_flood_exposure_analysis.ipynb`.
3. Run `notebooks/02_export_geojson.ipynb`.
4. Run `notebooks/04_sea_level_trend.ipynb`.
5. Run `notebooks/05_build_html.ipynb`.

The final build notebook generates both `web/index.html` and
`docs/index.html` from `web/index_template.html`.

Do not require the ASTER and WorldPop rasters to have the same native CRS or
resolution at this preprocessing stage; those expected differences are handled
explicitly downstream.
